In [ ]:
%pip install qiskit numpy qiskit-aer

# 🔬 Notebook 6: Quantum Teleportation

### *Ode to Quantum: Meridian Station Quantum Core Lab*

> The comms relay is down. Send the qubit anyway.

---

## 🛰️ Mission Briefing


Alert klaxon. "Cadet, the physical comms relay between Meridian Station and the
outpost has just failed. We have an unknown quantum state sitting on a single qubit
that absolutely must reach the outpost intact. We cannot physically move the qubit.
We cannot measure it and read off its value; measurement would destroy the very
information we're trying to send."

"Fortunately, we prepared for this. Entangled pairs were distributed to both
stations weeks ago, for exactly this scenario. Let's build the protocol that turns
that old entangled pair into an emergency channel, right now, piece by piece."

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Explain the quantum teleportation protocol step by step
- Build it yourself, one stage at a time, not as a single copy-pasted block
- Explain why teleportation requires a classical communication channel
- Verify, statistically, that the teleported state matches the original


## 🧩 Prerequisites

- Notebook 2: measurement and classical bits
- Notebook 5: entanglement and Bell pairs


## 💡 Concept


Quantum teleportation moves a qubit's exact state from one location to another
**without physically transporting the qubit**, using:

1. An entangled pair shared in advance between sender and receiver;
2. A joint measurement the sender performs on the unknown qubit *and* their half of
   the entangled pair (a **Bell measurement**);
3. Two classical bits of information sent from sender to receiver by ordinary
   classical means (radio, cable, whatever you've got);
4. A small correction the receiver applies, based on those two bits.

Nothing here moves information faster than light; step 3 is an ordinary classical
message, and the receiver's qubit is useless without it. What *does* move is the
qubit's exact state, without the original qubit ever leaving its location. We'll
build all four pieces in order.

## 📊 Visualization


Three qubits, three classical bits:

- **Qubit 0**: the message. Its state is unknown to us and must not be measured directly.
- **Qubit 1**: the sender's ("Alice's") half of a pre-shared entangled pair.
- **Qubit 2**: the receiver's ("Bob's") half of that same pair.

Let's build this up stage by stage.


## 🧪 Hands-on Code


**Stage 1: Prepare the unknown message.**

In a real scenario you wouldn't know this state; that's the whole point of
teleporting it rather than just telling Bob what it is. Here, we'll "pretend" not
to know it by picking a somewhat arbitrary angle, and we'll only "cheat" and look at
it directly for the sake of checking our work at the very end.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

theta = 1.9  # pretend this angle is unknown to whoever is teleporting the qubit

qc = QuantumCircuit(3, 3)
qc.ry(theta, 0)   # qubit 0 now holds the "unknown" message state
qc.barrier()

print(qc.draw())
print("(For our eyes only) message qubit's target probabilities:",
      f"P(0)={np.cos(theta/2)**2:.3f}  P(1)={np.sin(theta/2)**2:.3f}")


**Stage 2: Share an entangled pair between qubits 1 and 2.**

This is exactly the Bell pair you built in Notebook 5. In a real deployment, this
step would have happened long before the message even existed; the entangled pair
is prepared and distributed in advance, then held in reserve until it's needed.

In [ ]:
qc.h(1)
qc.cx(1, 2)
qc.barrier()

print(qc.draw())


**Stage 3: Bell measurement.**

Now the sender combines the message qubit (0) with their half of the entangled pair
(1) and measures both. This is called a Bell measurement because it measures in a
basis of entangled states rather than the plain `|0⟩`/`|1⟩` basis, and it's the
step that "uses up" the entanglement.

In [ ]:
qc.cx(0, 1)
qc.h(0)
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

print(qc.draw())


**Stage 4: Classical communication and correction.**

The two measurement results (`c0`, `c1`) are ordinary classical bits now. The
sender transmits them to the receiver by any classical channel. The receiver
applies a small correction to qubit 2, chosen based on those two bits:

In [ ]:
with qc.if_test((qc.clbits[1], 1)):
    qc.x(2)
with qc.if_test((qc.clbits[0], 1)):
    qc.z(2)

qc.measure(2, 2)

qc.draw("mpl")


## 📐 Math Lens


Why an `X` correction from bit `c1` and a `Z` correction from bit `c0`? The Bell
measurement in Stage 3 doesn't destroy the message; it reveals *which one of four
possible Bell states* the message and Alice's qubit collapsed into. Three of those
four outcomes leave Bob's qubit slightly "off" from the original message (flipped
(`X`), phase-flipped (`Z`), or both), and the two classical bits tell Bob exactly
which correction, if any, undoes that offset. Only the `00` outcome needs no
correction at all. Working out the full algebra behind that correspondence is a
great follow-up exercise, but the important result for now is: **two classical bits
are exactly enough to identify one of four possible corrections.**

## 🔁 Experiments


Let's verify the protocol actually worked statistically, since a single run only
gives you one bit of information (`0` or `1`) either way.


In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

backend = AerSimulator()
tqc = transpile(qc, backend)
counts = backend.run(tqc, shots=4000).result().get_counts()

# counts keys are bitstrings "c2 c1 c0" (Qiskit lists classical bits right-to-left)
# we only care about c2 -- Bob's final qubit, the teleportation target
q2_counts = {"0": 0, "1": 0}
for bitstring, n in counts.items():
    q2_counts[bitstring[0]] += n

total = sum(q2_counts.values())
measured_p0 = q2_counts["0"] / total
measured_p1 = q2_counts["1"] / total

print("Bob's qubit (qubit 2), measured over 4000 shots:")
print(f"  P(0) = {measured_p0:.3f}   (target: {np.cos(theta/2)**2:.3f})")
print(f"  P(1) = {measured_p1:.3f}   (target: {np.sin(theta/2)**2:.3f})")


Bob's qubit, which never once touched the original message qubit directly, and
which the sender never physically sent anywhere, reproduces the original state's
measurement statistics. The state teleported; the physical qubit did not.

## 🚀 Challenge


**Do it yourself, from scratch, with a different state.**

In the empty cell below, rebuild the entire protocol for a *different* angle of
your choosing (pick your own `theta2`; don't reuse `1.9`). Don't scroll up and
copy-paste; re-type each stage from memory, checking your reasoning against the
Concept section if you get stuck on the order of operations. At the end, run it for
4000 shots and confirm Bob's qubit matches your new angle's predicted
probabilities.

In [ ]:
# Your turn — teleport a state with a NEW angle, built stage by stage, from scratch.


## 🪞 Reflection


Nothing about this protocol lets you send information instantaneously; Bob's
qubit is meaningless static until the two classical bits arrive, and those bits
travel no faster than any ordinary signal.

What teleportation actually buys you is
subtler and, in its own way, more useful: an exact, unknown quantum state moved
from one location to another using only entanglement (prepared in advance) plus a
small classical message (sent when needed); without ever needing to know, measure,
or clone the state directly.

That last part matters more than it might seem: it's a
direct consequence of a deep result called the **no-cloning theorem**, which says
you can never make an independent copy of an unknown quantum state.

 Teleportation
doesn't get around that theorem; it works *because* of it: the original qubit's
state is destroyed by the Bell measurement at the exact moment it reappears on
Bob's qubit, so no copy is ever created.

## 📦 Summary

- Teleportation moves a qubit's state, not the qubit itself
- It requires a pre-shared entangled pair plus two classical bits sent afterward;
- The Bell measurement identifies which of four corrections Bob needs to apply
- No information travels faster than light; the classical bits are required
- Teleportation is compatible with, not a violation of, the no-cloning theorem

## ➡️ Next Mission


You've now built the full foundational track of Ode to Quantum: qubits, gates,
measurement, superposition, multi-qubit systems, entanglement, and teleportation.
The complete curriculum continues from here into Part III, quantum algorithms
(Deutsch, Grover's search, the Quantum Fourier Transform) and Part IV, variational
quantum computing, before finally arriving at Part V.

For this release, we're jumping straight to a preview of that final destination.
**Notebook 7: Intro to Quantum Machine Learning** is a taste of where the full
path leads. The notebooks in between are mapped out in the curriculum roadmap,
ready to be built next.


---
*End of transmission.*